## Agentic Loop
With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

An agent has three parts:

    - Instructions, the role and behavior we want. We pass this as the developer message. The better the instructions, the better the agent helps.
    - Tools, the functions the agent can call to carry out the task. For us that's only search.
    - Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

Everything below is the code that wires these three together inside a loop.

In [1]:
# Importing libraries and create openai client
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import requests

In [330]:
openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

model = "qwen/qwen3.5-9b"

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [331]:
# Developer Prompt - we define how we want the LLM model to behave
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):

1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.
   This will give you general information about the topic.

2. ANALYZE: Carefully read all results from the first search.
   
3. SECOND SEARCH: Based on what you found, perform a SECOND search with:
   - New keywords derived from the first results
   - More specific questions or follow-up queries
   - OR ask yourself: "What else should I search for to give a complete answer?"

4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.

5. ASK: End by asking if there are other areas the user wants to explore.

Make sure you perform at least 2 separate searches before answering.

Make sure you perform no more than 3 separate searches before answering.
""".strip()

question= "Can I still join the course?"

In [332]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [333]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [334]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [335]:
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
    parallel_tool_calls=True,
)

In [336]:
response

ChatCompletion(id='chatcmpl-8dvm4isd3bop1ktcadcolj', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='877042638', function=Function(arguments='{"query":"join the course late enrollment deadline register"}', name='search'), type='function')], reasoning_content=''))], created=1782077572, model='qwen/qwen3.5-9b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='qwen/qwen3.5-9b', usage=CompletionUsage(completion_tokens=30, prompt_tokens=508, total_tokens=538, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=None), stats={})

In [147]:
response.choices[0]

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='507307455', function=Function(arguments='{"query":"join course enrollment register"}', name='search'), type='function')], reasoning_content=''))

In [24]:
response.choices[0].message.tool_calls[0]

ChatCompletionMessageFunctionToolCall(id='256669733', function=Function(arguments='{"query":"join course late registration enroll"}', name='search'), type='function')

In [345]:
import json

call = response.choices[0].message.tool_calls[0]
args_dict = json.loads(call.function.arguments)

results = search(**args_dict)
result_json = json.dumps(results, indent=2)

In [201]:
call

ChatCompletionMessageFunctionToolCall(id='438478323', function=Function(arguments='{"query":"Can I still join the course? enrollment policies"}', name='search'), type='function')

In [202]:
call.function.name

'search'

In [203]:
args_dict

{'query': 'Can I still join the course? enrollment policies'}

In [204]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "9f689c185f",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I missed the first homework - can I still get a certificate?",
    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certifi

In [346]:
# Initial message history
messages

[{'role': 'system',
  'content': 'You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.\n\nMake sure you perform no more than 3 separate searches before answering.'},
 {'role': 'us

I need output tool call info and the tool call output

In [347]:
# Put all message history together
# Need to append LLM's tool call info to messages history
messages.append({
    "role": "tool",
    "arguments": args_dict,
    "tool_call_id": call.id,
    "name": call.function.name,
    "content": '',
})


In [348]:
messages

[{'role': 'system',
  'content': 'You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.\n\nMake sure you perform no more than 3 separate searches before answering.'},
 {'role': 'us

In [349]:
# Need to send search results to messages history as well
messages.append({
    "role": "tool",
    "tool_call_id": call.id, # for the response API you use call.call_id
    "content": result_json,
})

In [350]:
# Updated message history
messages # this info will be sent to the LLM

[{'role': 'system',
  'content': 'You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.\n\nMake sure you perform no more than 3 separate searches before answering.'},
 {'role': 'us

In [351]:
# Run once more
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
    parallel_tool_calls=True,
)

In [354]:
print(response.choices[0].finish_reason)

stop


Now how do we make multiple tool calls with the LLM model?

Basic steps:
1. make a call to the LLM <--
2. LLM decided to invoke search('params')
3. We invoke the search, we have the results
4. Send the results back to the LLM (another call) <--
5. LLM processes the results
6. LLM makes an additional call to use search again* (create a tool call loop until LLM model is ready to give answer)
7. Send the results additional results to the LLM (another call) <--
8. LLM processes the results
9. LLM gives the answer

This is call Agentic Loop

Parts to construct agentic looping:
- We give a role to the agent which includes instructions to use specific tools and call multiples as needed
- We have memory which is represented by our messages history so far in order to keep track of the previous interactions
- Finally we have the tools so the agent knows what tools it can use to fulfill the request from the user

We also include an exit loop which can implemented by checking if there are no more tool calls.

In [414]:
# Developer Prompt - we define how we want the LLM model to behave
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):

1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.
   This will give you general information about the topic.

2. ANALYZE: Carefully read all results from the first search.
   
3. SECOND SEARCH: Based on what you found, perform a SECOND search with:
   - New keywords derived from the first results
   - More specific questions or follow-up queries
   - OR ask yourself: "What else should I search for to give a complete answer?"

4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.

5. ASK: End by asking if there are other areas the user wants to explore.

Make sure you perform at least 2 separate searches before answering.
""".strip()

question= "Can I still join the course?"

In [415]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [416]:
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
    parallel_tool_calls=True,
)

In [417]:
response

ChatCompletion(id='chatcmpl-kskw5jk71i1ucengbfwrx', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='471452149', function=Function(arguments='{"query":"join course enroll start registration"}', name='search'), type='function')], reasoning_content=''))], created=1782081244, model='qwen/qwen3.5-9b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='qwen/qwen3.5-9b', usage=CompletionUsage(completion_tokens=28, prompt_tokens=493, total_tokens=521, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=None), stats={})

In [418]:
# Creating a helper call search function repeatedly
def make_call(call):
    """Convert JSON string to dict and invoke the search function."""
    args = json.loads(call.function.arguments)
    
    if call.function.name == "search":
        results = search(**args)
    
    result_json = json.dumps(results, indent=2)

    return {
    "role": "tool",
    "tool_call_id": call.id,
    "content": result_json,
    }


This make call function will takes the args from the response call and inputs them into the search function, calls the search function and finally return the search results which will be sent to the LLM model.

note: chat.completions api may not be able to store multiple calls to one variable

In [420]:
for item in response.choices:
    print(item)

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='471452149', function=Function(arguments='{"query":"join course enroll start registration"}', name='search'), type='function')], reasoning_content=''))


In [421]:
for item in response.choices:
    print(item.message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='471452149', function=Function(arguments='{"query":"join course enroll start registration"}', name='search'), type='function')]


In [422]:
for item in response.choices:
    print(item.message.tool_calls[0].function.name)

search


In [423]:
for item in response.choices:
    print(item.message.tool_calls[0].function.arguments)

{"query":"join course enroll start registration"}


In [424]:
# Process all tool calls
for item in response.choices:
    print(f"Call response: {item}")
    if item.message.tool_calls[0].type == 'function':
        print(f"Processing tool call: {item.message.tool_calls[0].function.name}")

        call_output = make_call(item.message.tool_calls[0])

        messages.append({
            "role": "tool",
            "arguments": item.message.tool_calls[0].function.arguments,
            "tool_call_id": item.message.tool_calls[0].id,
            "name": item.message.tool_calls[0].function.name,
            "content": item.message.content
        })

        messages.append(call_output)

    if item.finish_reason == 'stop':
        pass


Call response: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='471452149', function=Function(arguments='{"query":"join course enroll start registration"}', name='search'), type='function')], reasoning_content=''))
Processing tool call: search


In [425]:
messages

[{'role': 'system',
  'content': 'You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.'},
 {'role': 'user', 'content': 'Can I still join the course?'},
 {'role': 'tool',
  'argume

In [426]:
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
    parallel_tool_calls=True,
)

In [427]:
response

ChatCompletion(id='chatcmpl-d1fe7wgowhe30q1js1zp6w', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Yes, you can still join the course! As mentioned in our FAQ, anyone can start whenever they want. The videos and GitHub materials are available for you to access anytime.\n\nJust keep in mind that if you're interested in receiving a certificate, you'll need to submit your project while we're still accepting submissions (which typically happens during live cohorts). The current cohort runs until Summer 2027.\n\nYou can start by:\n1. Checking the course materials at https://datatalks.club/docs/courses/llm-zoomcamp/\n2. Reviewing the general Zoomcamp logistics docs\n3. Looking at the LLM Zoomcamp GitHub repository\n4. Following the typical workflow of watching videos, working through notebooks, and submitting homework\n\nWould you like me to help you with anything specific about getting started with the course?", refusal=None, role='assi

In [ ]:
for item in response.choices:
    if item.message.tool_calls[0].type == 'function':
        print(f"Processing tool call: {item.message.tool_calls[0].function.name}")
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)